In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
"""
misclassified_visualizer_v9.py
--------------------------------
Visualizes misclassified samples from MicroExpression model evaluation.
Loads cached frame tensors from .pt files.

Requirements:
- Ensure `load_sequences.py` is available in the microexpression folder for the dataset class.
- Dataset path matches training preprocessing.
- `misclassified_samples_v9.txt` should contain:
    Index: <idx>, True: <true_label>, Pred: <pred_label>

Author: Himanshu + ChatGPT
"""

import os
import importlib.util
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict
import torch
import numpy as np

# ===============================
# CONFIGURATION
# ===============================
DATASET_ROOT = "/content/drive/MyDrive/deepfake-detection/dataset/microexpression_processed"
CACHE_DIR = "/content/drive/MyDrive/deepfake-detection/microexpression/cache"
REPORTS_DIR = "/content/drive/MyDrive/deepfake-detection/microexpression/reports"
MISCLASSIFIED_FILE = os.path.join(REPORTS_DIR, "misclassified_samples_v9.txt")
PLOTS_SAVE_DIR = os.path.join(REPORTS_DIR, "misclassified_plots_v9")
PROJECT_ROOT = "/content/drive/MyDrive/deepfake-detection/microexpression"

os.makedirs(PLOTS_SAVE_DIR, exist_ok=True)

# ===============================
# Load MicroExpressionDataset from load_sequences.py
# ===============================
load_seq_path = os.path.join(PROJECT_ROOT, "load_sequences.py")

if not os.path.exists(load_seq_path):
    raise FileNotFoundError(f"❌ load_sequences.py not found at {load_seq_path}")

spec = importlib.util.spec_from_file_location("load_sequences", load_seq_path)
load_sequences = importlib.util.module_from_spec(spec)
spec.loader.exec_module(load_sequences)

MicroExpressionDataset = load_sequences.MicroExpressionDataset
print("📂 MicroExpressionDataset imported successfully!")

# ===============================
# Helper function to load cached frames (.pt)
# ===============================
def load_sequence(video_path):
    vid_id = os.path.splitext(os.path.basename(video_path))[0]
    cache_file = os.path.join(CACHE_DIR, f"{vid_id}.pt")
    if os.path.exists(cache_file):
        frames = torch.load(cache_file)
        return frames
    else:
        raise FileNotFoundError(f"Cache file not found for video {video_path}")

# ===============================
# LOAD DATASET
# ===============================
print("📂 Initializing MicroExpressionDataset...")
dataset = MicroExpressionDataset(
    root_dir=DATASET_ROOT,
    cache_dir=CACHE_DIR
)
print(f"✅ Dataset loaded: {len(dataset)} samples")

# ===============================
# Manual class names for fallback
# ===============================
class_names = ['Anger', 'Disgust', 'Fear', 'Happiness', 'Sadness', 'Surprise']
idx_to_class = {i: name for i, name in enumerate(class_names)}
print(f"⚠️ Using manual class names: {idx_to_class}")

# ===============================
# READ MISCLASSIFIED FILE
# ===============================
print(f"📄 Reading misclassified samples from: {MISCLASSIFIED_FILE}")
misclassified_by_class = defaultdict(list)

with open(MISCLASSIFIED_FILE, "r") as f:
    for line in f:
        if "True:" in line and "Pred:" in line:
            try:
                idx = int(line.split(",")[0].split(":")[1].strip())
                true_label = int(line.split("True:")[1].split(",")[0].strip())
                pred_label = int(line.split("Pred:")[1].strip())
                misclassified_by_class[true_label].append((idx, true_label, pred_label))
            except Exception as e:
                print(f"⚠️ Could not parse line: {line.strip()} | Error: {e}")

print("🔍 Misclassified sample counts:", {cls: len(samples) for cls, samples in misclassified_by_class.items()})

# ===============================
# FUNCTION: PLOT SAMPLES
# ===============================
def plot_samples(sample_indices, title, save_path):
    """Plots a set of samples in a grid from dataset indices."""
    if not sample_indices:
        print(f"⚠️ No samples to plot for {title}")
        return

    plt.figure(figsize=(15, 6))
    ncols = min(5, len(sample_indices))
    nrows = 2

    for i, (idx, t_label, p_label) in enumerate(sample_indices):
        try:
            sample_info = dataset.samples[idx]
            video_path = sample_info[0] if isinstance(sample_info, tuple) else sample_info
            vid_id = os.path.splitext(os.path.basename(video_path))[0]

            frames_loaded = load_sequence(video_path)

            # Handle tuple output (e.g., (frames_tensor, other_info))
            if isinstance(frames_loaded, tuple):
                frames = frames_loaded[0]
            else:
                frames = frames_loaded

            if torch.is_tensor(frames):
                if frames.dim() == 4:  # N,C,H,W
                    frames = frames.permute(0, 2, 3, 1).cpu().numpy()
                    mid_frame = frames[len(frames) // 2]
                elif frames.dim() == 3:  # C,H,W single frame
                    mid_frame = frames.permute(1, 2, 0).cpu().numpy()
                elif frames.dim() == 2:  # H,W grayscale single frame
                    mid_frame = frames.cpu().numpy()
                else:
                    print(f"⚠️ Unexpected tensor dimension {frames.dim()} for frames at idx {idx}")
                    continue
            else:
                print(f"⚠️ Frames loaded are not a tensor at idx {idx}, got {type(frames)}")
                continue

            img = Image.fromarray(mid_frame.astype(np.uint8))

            plt.subplot(nrows, ncols, i + 1)
            plt.imshow(img)
            plt.axis("off")
            plt.title(f"T:{idx_to_class[t_label]}\nP:{idx_to_class[p_label]}", fontsize=8)

        except Exception as e:
            print(f"⚠️ Could not load index {idx}: {e}")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"💾 Saved plot: {save_path}")

# ===============================
# PLOT FOR EACH CLASS
# ===============================
for cls in range(len(idx_to_class)):
    title_mis = f"Class {cls} - Misclassified"
    save_mis = os.path.join(PLOTS_SAVE_DIR, f"class_{cls}_misclassified.png")
    plot_samples(misclassified_by_class.get(cls, []), title_mis, save_mis)

print(f"✅ Inspection complete. Plots saved to: {PLOTS_SAVE_DIR}")


📂 MicroExpressionDataset imported successfully!
📂 Initializing MicroExpressionDataset...
📂 Initializing MicroExpressionDataset...
🗂 Cache directory set to: /content/drive/MyDrive/deepfake-detection/microexpression/cache
🔍 Scanning dataset folders...
✅ Found 220 video samples across 6 emotion categories.
✅ Dataset loaded: 220 samples
⚠️ Using manual class names: {0: 'Anger', 1: 'Disgust', 2: 'Fear', 3: 'Happiness', 4: 'Sadness', 5: 'Surprise'}
📄 Reading misclassified samples from: /content/drive/MyDrive/deepfake-detection/microexpression/reports/misclassified_samples_v9.txt
🔍 Misclassified sample counts: {0: 2, 1: 1, 4: 2, 5: 4}
💾 Saved plot: /content/drive/MyDrive/deepfake-detection/microexpression/reports/misclassified_plots_v9/class_0_misclassified.png
💾 Saved plot: /content/drive/MyDrive/deepfake-detection/microexpression/reports/misclassified_plots_v9/class_1_misclassified.png
⚠️ No samples to plot for Class 2 - Misclassified
⚠️ No samples to plot for Class 3 - Misclassified
💾 Save